# Scenarios

In [1]:
import itertools
import pandas as pd
import plotly.express as px

from ambdes import SimConfig, Runner, lognormal_sd_from_mean_p90

In [2]:
# Scenarios
vehicle_hours = [40_000, 52_000, 64_000]
incidents_per_day = [800, 1000, 1200]
handover_means = [20, 35, 50]

In [3]:
results = []

for vh, ipd, hm in itertools.product(vehicle_hours, incidents_per_day, handover_means):

    # Recompute inter-arrival times proportionally across C1-C4
    category_splits = {"C1": 0.10, "C2": 0.50, "C3": 0.30, "C4": 0.10}
    mean_iat = {
        cat: 1440 / (ipd * split)
        for cat, split in category_splits.items()
    }

    # Recompute handover SD for new mean
    hm_sd = lognormal_sd_from_mean_p90(mean=hm, p90=hm * 2.5)

    ambsys_data = {
        "mean_iat_min": mean_iat,
        "mean_handover_time_min": hm,
        "sd_handover_time_min": hm_sd,
        "p90_handover_time_min": hm * 2.5,
    }

    config = SimConfig(
        ambsys_data=ambsys_data,
        resource_hours_per_week=vh,
        data_collection_period=7 * 1440,
        n_reps=3,
    )
    output = Runner(config).run_reps()

    df = output["overall"].copy()
    df["vehicle_hours"] = vh
    df["incidents_per_day"] = ipd
    df["handover_mean_min"] = hm
    results.append(df)

scenario_results = pd.concat(results, ignore_index=True)

/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary, utilisation], ignore_index=True)
/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([summary, utilisation], ignore_index=True)
/home/amy/Documents/ambulance/ambdes/src/ambdes/results.py:212: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecate

In [4]:
scenario_results

,category,mean_n_patients,mean_response_time,mean_utilisation,vehicle_hours,incidents_per_day,handover_mean_min
0,C1,570.666667,9.892654,NaN,40000,800,20
1,C2,2787.0,9.945532,NaN,40000,800,20
2,C3,1668.0,10.108811,NaN,40000,800,20
3,C4,582.666667,9.693527,NaN,40000,800,20
4,all,NaN,NaN,0.231909,40000,800,20
...,...,...,...,...,...,...,...
130,C1,847.0,9.800842,NaN,64000,1200,50
131,C2,4198.0,10.011756,NaN,64000,1200,50
132,C3,2502.666667,9.946042,NaN,64000,1200,50
133,C4,862.666667,9.798497,NaN,64000,1200,50


In [5]:
# Create a readable scenario label from the parameter columns
scenario_results["scenario"] = (
    "vh=" + scenario_results["vehicle_hours"].astype(str)
    + " | inc=" + scenario_results["incidents_per_day"].astype(str)
    + " | ho=" + scenario_results["handover_mean_min"].astype(str)
)

# Plot 1: Mean response time by category and scenario
fig1 = px.bar(
    scenario_results,
    x="category",
    y="mean_response_time",
    color="scenario",
    barmode="group",
    title="Mean Response Time by Category and Scenario",
    labels={
        "mean_response_time": "Mean Response Time (mins)",
        "category": "Response Category",
        "scenario": "Scenario",
    },
)
fig1.show()

# Mean patient count by category and scenario
fig2 = px.bar(
    scenario_results,
    x="category",
    y="mean_n_patients",
    color="scenario",
    barmode="group",
    title="Mean Patient Count by Category and Scenario",
    labels={
        "mean_n_patients": "Mean Number of Patients",
        "category": "Response Category",
        "scenario": "Scenario",
    },
)
fig2.show()

# Response time as a line across handover times, faceted by category
fig3 = px.line(
    scenario_results,
    x="handover_mean_min",
    y="mean_response_time",
    color="category",
    facet_col="incidents_per_day",
    line_group="vehicle_hours",
    markers=True,
    title="Effect of Handover Time on Response Time",
    labels={
        "handover_mean_min": "Mean Handover Time (mins)",
        "mean_response_time": "Mean Response Time (mins)",
        "category": "Category",
    },
)
fig3.show()